# scanner_ag_v1 — Lab Environment (Google Colab)

Este notebook reproduz o lab do capítulo em ambiente cloud gratuito.  
Útil se sua máquina local não tem RAM suficiente para rodar o modelo.

**Runtime recomendado:** `Runtime → Change runtime type → T4 GPU`  
Isso garante 15GB de VRAM + ~12GB de RAM de sistema — suficiente para `qwen3.5:9b`.

Siga as células em ordem.

## 1. Instalar Ollama no runtime

In [ ]:
# Instala dependência de sistema e Ollama
!apt-get install -qq -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Iniciar o servidor Ollama em background

In [ ]:
import subprocess
import time
import requests

proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(20):
    try:
        requests.get("http://localhost:11434", timeout=1)
        print("ollama serve: ready")
        break
    except Exception:
        time.sleep(1)
else:
    print("ollama serve: timeout — tente rodar a célula novamente")

## 3. Ativar keepalive (evita timeout por inatividade)

In [ ]:
import threading
import time
from IPython.display import display, Javascript

def keep_alive():
    while True:
        time.sleep(30)
        display(Javascript('null'))

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("keepalive: ativo")

## 4. Baixar o modelo

In [ ]:
# CPU only / 8GB RAM: qwen3.5:4b
# T4 GPU  / 16GB RAM: qwen3.5:9b
MODEL = "qwen3.5:4b"

!ollama pull {MODEL}

## 5. Instalar dependências Python

In [ ]:
!pip install -q openai tree-sitter tree-sitter-python

## 6. Upload dos arquivos

Selecione `scanner_ag_v1.py` e `cipher_v3.py` juntos segurando `Ctrl` no diálogo de seleção.

In [ ]:
from google.colab import files

print("Selecione scanner_ag_v1.py e cipher_v3.py (use Ctrl+clique para selecionar os dois)")
uploaded = files.upload()

for name in uploaded:
    print(f"uploaded: {name} ({len(uploaded[name])} bytes)")

## 7. Executar o scanner

In [ ]:
# Saída texto — legível para revisão
!python3 scanner_ag_v1.py cipher_v3.py --format=text --model={MODEL}

In [ ]:
# Com verbose — expõe as quatro fases em tempo real
!python3 scanner_ag_v1.py cipher_v3.py --format=text --verbose --model={MODEL}

In [ ]:
# Saída SARIF — machine-readable, compatível com GitHub Advanced Security
!python3 scanner_ag_v1.py cipher_v3.py --format=sarif --model={MODEL}

## Notas

- O runtime do Colab é efêmero: ao reconectar, repita as células 1–6.
- Para analisar outros arquivos, faça upload na célula 6 e ajuste o nome do alvo na célula 7.
- O modelo fica cacheado no runtime enquanto a sessão estiver ativa — chamadas subsequentes são mais rápidas.
- **Atenção:** não use este notebook para analisar código com segredos reais —
  o runtime do Google tem acesso aos arquivos durante a sessão.